In [ ]:
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import os
from datetime import date, timedelta

load_dotenv('../.env')
TOKEN = os.getenv('ACCESS_TOKEN')

HEADERS = {"Authorization": f"Bearer {TOKEN}"}
BASE_URL = "https://api.ouraring.com/v2/usercollection"
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def fetch_oura(endpoint: str, params: dict) -> list:
    """Fetch all pages from an Oura V2 endpoint."""
    results = []
    url = f"{BASE_URL}/{endpoint}"

    while url:
        r = requests.get(url, headers=HEADERS, params=params)
        r.raise_for_status()
        data = r.json()
        results.extend(data.get("data", []))
        url = data.get("next_token")  # pagination
        params = {}                   # only pass params on first request

    return results

In [ ]:
START_DATE = "2026-01-01" 
END_DATE   = "2026-04-10"  

In [ ]:
readiness_raw = fetch_oura(
    "daily_readiness",
    params={"start_date": START_DATE, "end_date": END_DATE}
)

df_readiness = pd.json_normalize(readiness_raw)
df_readiness.to_csv(RAW_DIR / "readiness.csv", index=False)
print(f"Readiness: {len(df_readiness)} rows → data/raw/readiness.csv")
print(df_readiness[["day", "temperature_deviation", "temperature_trend_deviation"]].head())

In [ ]:
def fetch_oura_heartrate(start: str, end: str) -> list:
    """Fetch heart rate in 30-day chunks."""
    results = []
    start_dt = date.fromisoformat(start)
    end_dt   = date.fromisoformat(end)

    while start_dt < end_dt:
        chunk_end = min(start_dt + timedelta(days=29), end_dt)
        print(f"Fetching {start_dt} → {chunk_end}...")

        r = requests.get(
            f"{BASE_URL}/heartrate",
            headers=HEADERS,
            params={
                "start_datetime": f"{start_dt}T00:00:00",
                "end_datetime":   f"{chunk_end}T23:59:59"
            }
        )
        r.raise_for_status()
        results.extend(r.json().get("data", []))
        start_dt = chunk_end + timedelta(days=1)

    return results

hr_raw = fetch_oura_heartrate(START_DATE, END_DATE)
df_hr  = pd.json_normalize(hr_raw)
df_hr.to_csv(RAW_DIR / "heart_rate.csv", index=False)
print(f"Heart rate: {len(df_hr)} rows → data/raw/heart_rate.csv")

In [ ]:
sleep_raw = fetch_oura(
    "sleep",
    params={"start_date": START_DATE, "end_date": END_DATE}
)

df_sleep = pd.json_normalize(sleep_raw)
df_sleep.to_csv(RAW_DIR / "sleep.csv", index=False)

In [ ]:
print("=== Readiness columns ===")
print(df_readiness.columns.tolist())

print("\n=== Heart rate columns ===")
print(df_hr.columns.tolist())

print("\n=== Date range in readiness ===")
print(df_readiness["day"].min(), "→", df_readiness["day"].max())

print("\n=== Sleep columns ===")
print(df_sleep.columns.tolist())